In [1]:
%load_ext autoreload
%autoreload 2

import sys, os
_root = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_root, "config.py")) and os.path.dirname(_root) != _root:
    _root = os.path.dirname(_root)
if _root not in sys.path: sys.path.insert(0, _root)

import numpy as np, pandas as pd, pyarrow.dataset as pds
import config
from cprics.io_data import load_peak_amplitudes
from cprics.continuous import build_all_combined, CANDIDATE_FLOOR
from cprics.tracking import track_islands

dset = pds.dataset(config.PEAK_AMP_DIR, partitioning="hive")
print([n for n in dset.schema.names])   # find the 4.5 / 6 / 13.5 / 18 top-right columns

['CT_Index', 'Mileage', 'Top_Right35', 'Top_Left35', '45_top_right', '45_top_left', '6_top_right', '6_top_left', '9_top_right', '9_top_left', '135_top_right', '135_top_left', '18_top_right', '18_top_left', 'Distance', 'Ref', 'Location_m', 'Location_Y', 'Date', 'ELR', 'TrackID']


In [2]:
NINE = "9_top_right"
isl9 = track_islands(build_all_combined(load_peak_amplitudes(config.PEAK_AMP_DIR, NINE),
                                        wavelength=9.0, candidate_floor=CANDIDATE_FLOOR))
foot = (isl9.groupby(["site_id", "ELR", "TrackID"])
            .agg(start=("start_location", "min"), end=("end_location", "max"))
            .reset_index())
foot = foot.merge(isl9.groupby("site_id")["hard_amplitude"].max().ge(30).rename("critical"),
                  on="site_id")
print("9m sites:", len(foot), " critical:", int(foot.critical.sum()))

9m sites: 2973  critical: 31


In [5]:
WL = {"45": 4.5, "6": 6.0, "9": 9.0, "135": 13.5, "18": 18.0}   # NR drops the decimal

TOL = 5.0
cov = foot.copy()
for wl, chan in BANDS.items():
    islb = build_all_combined(load_peak_amplitudes(config.PEAK_AMP_DIR, chan),
                              wavelength=WL[wl], candidate_floor=CANDIDATE_FLOOR)
    j = cov.merge(islb[["ELR", "TrackID", "center_location"]], on=["ELR", "TrackID"], how="left")
    hit = j[(j.center_location >= j.start - TOL) & (j.center_location <= j.end + TOL)]
    cov[f"has_{wl}"] = cov["site_id"].isin(hit["site_id"].unique())

bands = list(BANDS)
summary = pd.DataFrame({
    "wavelength_m": [WL[wl] for wl in bands],
    "all_sites":    [cov[f"has_{wl}"].mean() for wl in bands],
    "critical":     [cov.loc[cov.critical, f"has_{wl}"].mean() for wl in bands],
}, index=bands).round(3)
print(summary.to_string())

short = [wl for wl in bands if WL[wl] < 9]
cov["has_short"] = cov[[f"has_{wl}" for wl in short]].any(axis=1)
print(f"\nshort bands {[WL[wl] for wl in short]}:  all={cov.has_short.mean():.1%}   "
      f"critical={cov.loc[cov.critical, 'has_short'].mean():.1%}")

     wavelength_m  all_sites  critical
45            4.5      0.245     0.742
6             6.0      0.596     1.000
135          13.5      0.646     1.000
18           18.0      0.531     0.968

short bands [4.5, 6.0]:  all=60.5%   critical=100.0%


In [6]:
WLALL = {"45": 4.5, "6": 6.0, "9": 9.0, "135": 13.5, "18": 18.0}
CHAN  = {"45": "45_top_right", "6": "6_top_right", "9": NINE,
         "135": "135_top_right", "18": "18_top_right"}

def band_max_energy(chan, wl):
    islb = build_all_combined(load_peak_amplitudes(config.PEAK_AMP_DIR, chan),
                              wavelength=wl, candidate_floor=CANDIDATE_FLOOR)
    j = foot.merge(islb[["ELR", "TrackID", "center_location", "amplitude"]],
                   on=["ELR", "TrackID"], how="left")
    j = j[(j.center_location >= j.start - TOL) & (j.center_location <= j.end + TOL)]
    return j.groupby("site_id")["amplitude"].max()      # worst co-located energy in this band

E = foot.set_index("site_id")[["critical"]].copy()
for k, wl in WLALL.items():
    E[f"e_{wl}"] = band_max_energy(CHAN[k], wl).reindex(E.index)
E = E.fillna(0.0)

tot = E[[f"e_{w}" for w in [4.5, 6.0, 9.0, 13.5, 18.0]]].sum(axis=1).replace(0, np.nan)
E["short_frac"] = (E["e_4.5"] + E["e_6.0"]) / tot
E["off9_frac"]  = (tot - E["e_9.0"]) / tot
E["n_bands"]    = (E[["e_4.5", "e_6.0", "e_13.5", "e_18.0"]] > CANDIDATE_FLOOR).sum(axis=1)
E = E.fillna(0.0)

traj = isl9.groupby(["site_id", "date"])["amplitude"].max().reset_index().sort_values("date")
nine = traj.groupby("site_id").agg(current_amp=("amplitude", "last"), max_amp=("amplitude", "max"))
keep = traj.groupby("site_id").size(); keep = keep[keep >= 3].index
D = nine.join(E); D = D[D.index.isin(keep)]
print("sites:", len(D), " critical:", int(D.critical.sum()))
print(D[["short_frac", "off9_frac", "n_bands"]].describe().round(2).to_string())

sites: 2263  critical: 22
       short_frac  off9_frac  n_bands
count     2263.00    2263.00  2263.00
mean         0.14       0.51     0.69
std          0.21       0.32     0.96
min          0.00       0.00     0.00
25%          0.00       0.25     0.00
50%          0.04       0.56     0.00
75%          0.19       0.78     1.00
max          0.98       1.00     4.00


In [7]:
from sklearn.metrics import average_precision_score
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

y = D["critical"].astype(int).values
def loo(cols, label):
    pipe = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=2000))
    p = cross_val_predict(pipe, D[cols].values, y, cv=LeaveOneOut(), method="predict_proba")[:, 1]
    yk = y[np.argsort(-p)]
    print(f"{label:30} AUPRC {average_precision_score(y, p):.3f}  "
          f"P@20 {yk[:20].mean():.3f}  P@50 {yk[:50].mean():.3f}  R@50 {yk[:50].sum()/y.sum():.3f}")

loo(["current_amp", "max_amp"],                                      "base (9m amplitude)")
loo(["current_amp", "max_amp", "short_frac"],                       "+ short_frac (jaggedness)")
loo(["current_amp", "max_amp", "short_frac", "off9_frac", "n_bands"], "+ full texture")

base (9m amplitude)            AUPRC 0.853  P@20 0.750  P@50 0.440  R@50 1.000
+ short_frac (jaggedness)      AUPRC 0.860  P@20 0.800  P@50 0.440  R@50 1.000
+ full texture                 AUPRC 0.854  P@20 0.800  P@50 0.440  R@50 1.000


In [8]:
CONCERN = 18.0
NEIGH = [4.5, 6.0, 13.5, 18.0]
band_isl = {wl: build_all_combined(load_peak_amplitudes(config.PEAK_AMP_DIR, CHAN[k]),
                                   wavelength=wl, candidate_floor=CANDIDATE_FLOOR)
                 [["ELR", "TrackID", "date", "center_location", "amplitude"]]
            for k, wl in WLALL.items() if wl in NEIGH}

def first_concern_date(s, wl):
    b = band_isl[wl]
    m = b[(b.ELR == s.ELR) & (b.TrackID == s.TrackID) &
          (b.center_location >= s.start - TOL) & (b.center_location <= s.end + TOL)]
    t = m.groupby("date")["amplitude"].max()
    t = t[t >= CONCERN]
    return t.index.min() if len(t) else pd.NaT

rows = []
for _, s in foot[foot.critical].iterrows():
    own = isl9[isl9.site_id == s.site_id]
    hc = own.groupby("date")["hard_amplitude"].max(); t_crit = hc[hc >= 30].index.min()
    oc = own.groupby("date")["amplitude"].max();      flag9  = oc[oc >= CONCERN].index.min()
    firsts = {wl: first_concern_date(s, wl) for wl in NEIGH}
    firsts["9m"] = flag9
    firsts = {k: v for k, v in firsts.items() if pd.notna(v)}
    flag_mb = min(firsts.values()) if firsts else pd.NaT
    rows.append({"site_id": s.site_id, "t_crit": t_crit, "flag9": flag9, "flag_mb": flag_mb,
                 "first_band": min(firsts, key=firsts.get) if firsts else None})

R = pd.DataFrame(rows)
R["lead9"]   = (R.t_crit - R.flag9).dt.days / 30.44
R["lead_mb"] = (R.t_crit - R.flag_mb).dt.days / 30.44
R["adv9"]    = R.flag9 < R.t_crit
R["adv_mb"]  = R.flag_mb < R.t_crit

print(f"9m-only:   advance {R.adv9.mean():.1%}   median lead {R.lead9[R.adv9].median():.1f} mo")
print(f"multiband: advance {R.adv_mb.mean():.1%}   median lead {R.lead_mb[R.adv_mb].median():.1f} mo")
sudden = R[~R.adv9]
print(f"\nsudden-to-9m: {len(sudden)} of {len(R)} critical sites")
print(f"  rescued by a neighbour band (now flagged early): {int(sudden.adv_mb.sum())}")
print("\nwhich band fires first:\n" + R.first_band.value_counts().to_string())

9m-only:   advance 45.2%   median lead 11.5 mo
multiband: advance 61.3%   median lead 12.9 mo

sudden-to-9m: 17 of 31 critical sites
  rescued by a neighbour band (now flagged early): 5

which band fires first:
first_band
9m      21
13.5     5
6.0      3
18.0     2


1. The uniform 18 mm threshold is rigged toward the bigger bands. Longer wavelengths accumulate more, so they clear a fixed 18 mm line more easily — which is exactly why 13.5 m fires first most often (5) and 4.5 m never fires at all. That’s almost certainly a scale artifact, not “13.5 m is the best precursor.” The short “jaggedness” bands the rail team cares about are being penalised by the threshold, so we genuinely can’t say yet whether they carry early signal.
2. 13.5 m is the band immediately adjacent to 9 m, so some of its “early” crossing could be Butterworth passband bleed between neighbours rather than an independent precursor — the overlap question we flagged for Andy now matters directly.
3. This ran on 31 critical sites (all tracked), not the ≥3-obs 22 that exp02’s 54.5% was measured on, so the 45.2% isn’t directly comparable — the extra 9 short-history sites are the ones most likely to look “sudden.”

before believing any of it, the fair test: give each band its own threshold calibrated to the same selectivity as 9 m @ 18 (each band flags its top ~4% of sites), so “fires first” means “high for that band,” not “bigger band crosses a fixed line first.” And measure the cost — earlier flags are only worth it if they don’t flood the list.

In [9]:
crit = foot[foot.critical & foot.site_id.isin(keep)]     # the 22-set, comparable to exp02
p = (E["e_9.0"] >= CONCERN).mean()                        # 9m flag rate in the population
theta = {9.0: CONCERN, **{wl: float(E[f"e_{wl}"].quantile(1 - p)) for wl in NEIGH}}
print("9m flag rate:", round(p, 3), " per-band thresholds:", {k: round(v, 1) for k, v in theta.items()})

def first_cross(s, wl, thr):
    b = band_isl[wl]
    m = b[(b.ELR == s.ELR) & (b.TrackID == s.TrackID) &
          (b.center_location >= s.start - TOL) & (b.center_location <= s.end + TOL)]
    t = m.groupby("date")["amplitude"].max(); t = t[t >= thr]
    return t.index.min() if len(t) else pd.NaT

rows = []
for _, s in crit.iterrows():
    own = isl9[isl9.site_id == s.site_id]
    hc = own.groupby("date")["hard_amplitude"].max(); t_crit = hc[hc >= 30].index.min()
    oc = own.groupby("date")["amplitude"].max();      flag9  = oc[oc >= CONCERN].index.min()
    firsts = {wl: first_cross(s, wl, theta[wl]) for wl in NEIGH}; firsts["9m"] = flag9
    firsts = {k: v for k, v in firsts.items() if pd.notna(v)}
    fmb = min(firsts.values()) if firsts else pd.NaT
    rows.append({"adv9": flag9 < t_crit, "adv_mb": (fmb < t_crit) if pd.notna(fmb) else False,
                 "first_band": min(firsts, key=firsts.get) if firsts else None})
R2 = pd.DataFrame(rows)
print(f"\n9m-only:   advance {R2.adv9.mean():.1%}")
print(f"multiband: advance {R2.adv_mb.mean():.1%}   "
      f"rescued sudden: {int((~R2.adv9 & R2.adv_mb).sum())} of {int((~R2.adv9).sum())}")
print("first band:\n" + R2.first_band.value_counts().to_string())

flag9_all  = (E["e_9.0"] >= CONCERN)
flagmb_all = flag9_all.copy()
for wl in NEIGH: flagmb_all |= (E[f"e_{wl}"] >= theta[wl])
print(f"\nwatchlist: 9m-only {int(flag9_all.sum())} -> multiband {int(flagmb_all.sum())} "
      f"(+{int(flagmb_all.sum() - flag9_all.sum())}, {flagmb_all.sum()/flag9_all.sum():.1f}x)")

9m flag rate: 0.059  per-band thresholds: {9.0: 18.0, 4.5: 0.7, 6.0: 5.0, 13.5: 13.6, 18.0: 12.4}

9m-only:   advance 54.5%
multiband: advance 68.2%   rescued sudden: 3 of 10
first band:
first_band
4.5     9
9m      7
6.0     4
13.5    2

watchlist: 9m-only 176 -> multiband 521 (+345, 3.0x)


In [10]:
# normalise each band by its equal-selectivity threshold; multiband score = max_b(amp_b / theta_b)
budget = int(flag9_all.sum())                                  # 176, the 9m@18 cost
score_worst = pd.concat([E[f"e_{wl}"] / theta[wl] for wl in theta], axis=1).max(axis=1)
c = score_worst.sort_values(ascending=False).iloc[budget - 1]  # cutoff giving the same 176 sites
print(f"budget {budget} sites; multiband cutoff c={c:.2f}; realised size={int((score_worst>=c).sum())}")

def mb_flag_date(s, c):
    dates = (isl9[isl9.site_id == s.site_id].groupby("date")["amplitude"].max() / theta[9.0]).to_dict()
    for wl in NEIGH:
        b = band_isl[wl]
        m = b[(b.ELR == s.ELR) & (b.TrackID == s.TrackID) &
              (b.center_location >= s.start - TOL) & (b.center_location <= s.end + TOL)]
        for d, v in (m.groupby("date")["amplitude"].max() / theta[wl]).items():
            dates[d] = max(dates.get(d, 0), v)
    crossed = sorted(d for d, v in dates.items() if v >= c)
    return crossed[0] if crossed else pd.NaT

rows = []
for _, s in crit.iterrows():
    own = isl9[isl9.site_id == s.site_id]
    t_crit = (own.groupby("date")["hard_amplitude"].max() >= 30); t_crit = t_crit[t_crit].index.min()
    flag9 = (own.groupby("date")["amplitude"].max() >= CONCERN); flag9 = flag9[flag9].index.min()
    fmb = mb_flag_date(s, c)
    rows.append({"adv9": flag9 < t_crit, "adv_mb": (fmb < t_crit) if pd.notna(fmb) else False})
R3 = pd.DataFrame(rows)

critids = list(crit.site_id)
print(f"\nat equal budget ({budget} sites):")
print(f"  criticals captured: 9m {(E.loc[critids,'e_9.0']>=CONCERN).sum()}/{len(critids)}  "
      f"multiband {(score_worst.loc[critids]>=c).sum()}/{len(critids)}")
print(f"  9m-only   advance {R3.adv9.mean():.1%}")
print(f"  multiband advance {R3.adv_mb.mean():.1%}   "
      f"rescued sudden {int((~R3.adv9 & R3.adv_mb).sum())} of {int((~R3.adv9).sum())}")

budget 176 sites; multiband cutoff c=2.25; realised size=177

at equal budget (176 sites):
  criticals captured: 9m 22/22  multiband 18/22
  9m-only   advance 54.5%
  multiband advance 27.3%   rescued sudden 1 of 10


In [11]:
import json, os
findings = {
    "experiment": "exp03_multiband_9R", "verdict": "negative",
    "hypothesis": "short-wavelength/cross-band texture flags escalation or warns earlier than 9m alone",
    "coverage_pct_critical": {"4.5": 0.742, "6": 1.0, "13.5": 1.0, "18": 0.968},
    "coverage_pct_all":      {"4.5": 0.245, "6": 0.596, "13.5": 0.646, "18": 0.531},
    "ranking_auprc": {"base": 0.853, "plus_short_frac": 0.860, "full_texture": 0.854},
    "equal_budget": {"budget_sites": 176,
                     "critical_capture": {"9m": "22/22", "multiband": "18/22"},
                     "advance_pct": {"9m": 0.545, "multiband": 0.273},
                     "rescued_sudden": "1/10"},
    "reason": "all bands filtered from the same Top35 signal; correlated with 9m, little independent signal",
    "caveat": "tested vs hard-9m>=30 escalation label, not ride-quality; not disproven for ride-quality",
    "promoted": False, "notebook": "notebooks/research/10_exp03_multiband.ipynb",
}
os.makedirs(config.OUTPUTS_DIR / "exp03_multiband_9R", exist_ok=True)
with open(config.OUTPUTS_DIR / "exp03_multiband_9R" / "findings.json", "w") as fh:
    json.dump(findings, fh, indent=2)
print("wrote", config.OUTPUTS_DIR / "exp03_multiband_9R" / "findings.json")

wrote c:\Users\s2887048\Documents\network_rail\CPRICSv2\outputs\exp03_multiband_9R\findings.json
